In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

print("All imports done ✓")

All imports done ✓


In [7]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Referer": "https://www.magicbricks.com/",
}

locality_urls = {
    "Prahlad Nagar": "https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment&cityName=Ahmedabad&localityName=Prahlad-Nagar",
    "Bopal":         "https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment&cityName=Ahmedabad&localityName=Bopal",
    "Satellite":     "https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment&cityName=Ahmedabad&localityName=Satellite",
    "SBR":           "https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment&cityName=Ahmedabad&localityName=Sindhu-Bhavan-Road",
    "Thaltej":       "https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment&cityName=Ahmedabad&localityName=Thaltej",
    "Naranpura":     "https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment&cityName=Ahmedabad&localityName=Naranpura",
    "Chandkheda":    "https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment&cityName=Ahmedabad&localityName=Chandkheda",
    "Nikol":         "https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment&cityName=Ahmedabad&localityName=Nikol",
    "Khodiyar Nagar":"https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment&cityName=Ahmedabad&localityName=Khodiyar-Nagar",
    "Science City":  "https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment&cityName=Ahmedabad&localityName=Science-City",
}

print(f"{len(locality_urls)} localities loaded ✓")

10 localities loaded ✓


In [8]:
test_url = locality_urls["Bopal"] + "&page=1"
response = requests.get(test_url, headers=headers)
print("Status code:", response.status_code)

if response.status_code == 200:
    print("✓ Connected to MagicBricks successfully")
else:
    print("✗ Something went wrong — paste the status code here")

Status code: 200
✓ Connected to MagicBricks successfully


In [19]:
soup = BeautifulSoup(response.text, "html.parser")

cards = soup.find_all("div", class_="mb-srp__card")
print(f"Cards found: {len(cards)}")

Cards found: 30


In [20]:
card = cards[0]

# Title — contains BHK + locality both
title = card.find("h2", class_="mb-srp__card--title")
title_text = title.text.strip() if title else "NOT FOUND"

# Area
area = card.find("div", class_="mb-srp__card__summary--value")
area_text = area.text.strip() if area else "NOT FOUND"

# Price
price = card.find("div", class_="mb-srp__card__price--amount")
price_text = price.text.strip() if price else "NOT FOUND"

# Price per sqft — already calculated by MagicBricks
ppsf = card.find("div", class_="mb-srp__card__price--size")
ppsf_text = ppsf.text.strip() if ppsf else "NOT FOUND"

print("Title:          ", title_text)
print("Area:           ", area_text)
print("Price:          ", price_text)
print("Price per sqft: ", ppsf_text)

Title:           3 BHK Flat for Sale in Mahadev Glory, Bopal, Ahmedabad
Area:            1850 sqft
Price:           ₹87 Lac
Price per sqft:  ₹4703 per sqft


Bopal
Bopal
Bopal
Prahlad Nagar
Satellite


In [14]:
all_data = []
PAGES = 5

for locality_name, base_url in locality_urls.items():
    print(f"\nScraping: {locality_name}")
    count = 0

    for page in range(1, PAGES + 1):
        url = base_url + f"&page={page}"

        try:
            resp = requests.get(url, headers=headers, timeout=15)

            if resp.status_code != 200:
                print(f"  Page {page}: blocked ({resp.status_code}) — skipping")
                time.sleep(10)
                continue

            soup  = BeautifulSoup(resp.text, "html.parser")
            cards = soup.find_all("div", class_="mb-srp__card")

            if len(cards) == 0:
                print(f"  Page {page}: no listings — stopping this locality")
                break

            for card in cards:
                try:
                    title_tag = card.find("h2",  class_="mb-srp__card--title")
                    area_tag  = card.find("div", class_="mb-srp__card__summary--value")
                    price_tag = card.find("div", class_="mb-srp__card__price--amount")
                    ppsf_tag  = card.find("div", class_="mb-srp__card__price--size")

                    title_text = title_tag.text.strip() if title_tag else ""

                    price_lakh = clean_price(price_tag.text  if price_tag else "")
                    area_sqft  = clean_area(area_tag.text    if area_tag  else "")
                    ppsf       = clean_ppsf(ppsf_tag.text    if ppsf_tag  else "")
                    bhk        = extract_bhk(title_text)
                    locality_detail = extract_locality(title_text, fallback_locality=locality_name)

                    if price_lakh and area_sqft and ppsf:
                        all_data.append({
                            "locality":        locality_name,
                            "locality_detail": locality_detail,
                            "bhk":             bhk,
                            "price_lakh":      price_lakh,
                            "area_sqft":       area_sqft,
                            "price_per_sqft":  ppsf,
                        })
                        count += 1

                except:
                    pass

            print(f"  Page {page}: ✓  ({count} listings so far)")
            time.sleep(3)

        except Exception as e:
            print(f"  Page {page}: error — {e}")
            time.sleep(10)

    print(f"  Done — {count} listings from {locality_name}")

print(f"\n✓ Scraping complete — {len(all_data)} total listings")


Scraping: Prahlad Nagar
  Page 1: ✓  (30 listings so far)
  Page 2: ✓  (60 listings so far)
  Page 3: ✓  (87 listings so far)
  Page 4: ✓  (116 listings so far)
  Page 5: ✓  (145 listings so far)
  Done — 145 listings from Prahlad Nagar

Scraping: Bopal
  Page 1: ✓  (30 listings so far)
  Page 2: ✓  (60 listings so far)
  Page 3: ✓  (90 listings so far)
  Page 4: ✓  (119 listings so far)
  Page 5: ✓  (149 listings so far)
  Done — 149 listings from Bopal

Scraping: Satellite
  Page 1: ✓  (30 listings so far)
  Page 2: ✓  (59 listings so far)
  Page 3: ✓  (85 listings so far)
  Page 4: ✓  (111 listings so far)
  Page 5: ✓  (137 listings so far)
  Done — 137 listings from Satellite

Scraping: SBR
  Page 1: ✓  (24 listings so far)
  Page 2: ✓  (53 listings so far)
  Page 3: ✓  (83 listings so far)
  Page 4: ✓  (113 listings so far)
  Page 5: ✓  (143 listings so far)
  Done — 143 listings from SBR

Scraping: Thaltej
  Page 1: ✓  (27 listings so far)
  Page 2: ✓  (57 listings so far)
  Pag

In [15]:
df = pd.DataFrame(all_data)

# Remove unrealistic values
df = df[df["price_per_sqft"] > 1500]
df = df[df["price_per_sqft"] < 25000]
df = df[df["area_sqft"]      > 200  ]
df = df.dropna(subset=["price_lakh", "area_sqft", "price_per_sqft"])

df.to_csv("ahmedabad_clean.csv", index=False)
print(f"✓ Saved — {len(df)} clean listings")
print(df.head())

✓ Saved — 1132 clean listings
        locality locality_detail  bhk  price_lakh  area_sqft  price_per_sqft
0  Prahlad Nagar   Prahlad Nagar  4.0       271.0     1828.0          7200.0
1  Prahlad Nagar   Prahlad Nagar  4.0       400.0     4015.0          9963.0
2  Prahlad Nagar   Prahlad Nagar  4.0       285.0     2651.0         10751.0
3  Prahlad Nagar   Prahlad Nagar  4.0       271.0     2035.0          7200.0
4  Prahlad Nagar   Prahlad Nagar  4.0       271.0     1828.0          7200.0


In [16]:
print("=" * 55)
print("Q1: BEST VALUE — avg price per sqft by locality")
print("=" * 55)
q1 = df.groupby("locality")["price_per_sqft"].mean().round(0).sort_values()
print(q1.to_string())

print("\n")
print("=" * 55)
print("Q2: MARKET ACTIVITY — listing count by locality")
print("=" * 55)
q2 = df.groupby("locality").size().sort_values(ascending=False)
print(q2.to_string())

print("\n")
print("=" * 55)
print("Q3: DOMINANT BHK TYPE per locality")
print("=" * 55)
q3 = df.groupby(["locality","bhk"]).size().unstack(fill_value=0)
print(q3.to_string())

print("\n")
cheapest  = q1.index[0]
costliest = q1.index[-1]
premium   = round((q1.iloc[-1] - q1.iloc[0]) / q1.iloc[0] * 100, 1)
print("=" * 55)
print("YOUR RESUME LINE:")
print("=" * 55)
print(f"{costliest} is {premium}% more expensive per sqft than {cheapest}")

Q1: BEST VALUE — avg price per sqft by locality
locality
Nikol             4758.0
Khodiyar Nagar    4912.0
Chandkheda        4986.0
Bopal             5694.0
SBR               6198.0
Science City      6661.0
Naranpura         6878.0
Prahlad Nagar     7851.0
Thaltej           7993.0
Satellite         8603.0


Q2: MARKET ACTIVITY — listing count by locality
locality
Bopal             147
Prahlad Nagar     144
Thaltej           137
SBR               136
Naranpura         130
Science City      125
Satellite         118
Chandkheda        102
Nikol              75
Khodiyar Nagar     18


Q3: DOMINANT BHK TYPE per locality
bhk             1.0  2.0  3.0  4.0  5.0  6.0  7.0
locality                                         
Bopal             1   22   86   33    5    0    0
Chandkheda        6   41   48    6    0    0    0
Khodiyar Nagar    0    6    9    3    0    0    0
Naranpura         1    9   95   24    1    0    0
Nikol            15   33   21    6    0    0    0
Prahlad Nagar     3   20   

In [17]:
df_raw = pd.DataFrame(all_data)
df_raw.to_csv("ahmedabad.realestate2026_raw.csv", index=False)
print("Saved!")

Saved!
